# Import des bibliothèques nécessaires

In [1]:
import pandas as pd #pour la manipulation de données
import numpy as np
from pathlib import Path #pour la gestion des chemins de fichiers
import spacy #pour le prétraitement de texte
from spacy.lang.fr.stop_words import STOP_WORDS as spacy_stopwords


# Chargement des données

In [2]:
df=pd.read_csv(Path("..") /"data" /"2_processed"/ "01_paquets_128.csv", encoding="utf-8",)
df.head()

,nom_fichier,id_paquet,phrases_paquet
0,1893_20_Le_docteur_Pascal._clean.txt,0,Dans la chaleur de l’ardente après-midi de jui...
1,1893_20_Le_docteur_Pascal._clean.txt,1,Il y avait plus de trente ans que le docteur y...
2,1893_20_Le_docteur_Pascal._clean.txt,2,"Tiens! Clotilde, finit-il par dire, tu recopie..."
3,1893_20_Le_docteur_Pascal._clean.txt,3,"Dans sa longue blouse noire, elle était très g..."
4,1893_20_Le_docteur_Pascal._clean.txt,4,Des chaises et des fauteuils antiques traînaie...


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29421 entries, 0 to 29420
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   nom_fichier     29421 non-null  str  
 1   id_paquet       29421 non-null  int64
 2   phrases_paquet  29421 non-null  str  
dtypes: int64(1), str(2)
memory usage: 689.7 KB


# Préparation du dataframe 

In [4]:
df = df.rename(columns={
    "nom_fichier": "fichier",
    "id_paquet": "paquet_id",
    "phrases_paquet": "texte"
})

In [5]:
df["annee"] = df["fichier"].str.extract(r"^(\d{4})").astype(int)

In [6]:
ordre_romans= (
    df[["fichier", "annee"]]
    .drop_duplicates()
    .sort_values(["annee", "fichier"])
    .reset_index(drop=True)
)

ordre_romans["ordre_romans"] = range(1, len(ordre_romans) + 1)

df = df.merge(ordre_romans[["fichier", "ordre_romans"]], on="fichier", how="left")

In [7]:
df["roman"] = (
    df["fichier"]
    .str.replace(r"^\d{4}_\d+_", "", regex=True)
    .str.replace(r"_clean\.txt$", "", regex=True)
    .str.replace("_", " ")
)

In [8]:
df["nb_mots"] = df["texte"].str.split().str.len()

In [9]:
df = df[[
    "roman",
    "annee",
    "ordre_romans",
    "paquet_id",
    "texte",
    "nb_mots"
]]

In [10]:
df = df.sort_values(["ordre_romans", "paquet_id"]).reset_index(drop=True)

In [11]:
df["paquet_id"] = df.groupby("roman").cumcount() + 1

In [12]:
df.head()

,roman,annee,ordre_romans,paquet_id,texte,nb_mots
0,1865 La confession de Claude.,1865,1,1,"Voici l’hiver: l’air, au matin, devient plus f...",145
1,1865 La confession de Claude.,1865,1,2,"Le soir, quand le vent ébranle la porte et que...",135
2,1865 La confession de Claude.,1865,1,3,"Et, lorsque ma chambre ne veut pour sourire qu...",133
3,1865 La confession de Claude.,1865,1,4,Tous trois nous laissions nos lèvres dire ce q...,130
4,1865 La confession de Claude.,1865,1,5,"Les vôtres, vous souvenez-vous? brunes et rieu...",149


In [14]:
chemin_sortie = Path("..") /"data" /"2_processed" /"02_corpus_zola_128.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")

# Lemmatisation des données

In [15]:
stop_perso = {
    "grand", "petit", "homme", "femme", "jour", "heure", "coup", "œil", "oeil", 
    "main", "bras", "tête", "voix", "milieu", "eau", "terre", "air", "monde", 
    "chose", "nuit", "vie", "enfant", "père", "mère", "fille", "garçon", 
    "monsieur", "madame", "falloir", "aller", "voir", "dire", "faire", 
    "pouvoir", "vouloir", "savoir", "venir", "devoir", "prendre", "donner",
    "oui", "non", "où", "quand", "comment", "bon", "jeune", "vieux", "suite"
}

# Chargement du modèle avec désactivation du 'parser' syntaxique pour gagner en vitesse
# On garde impérativement 'ner' pour repérer les personnages/lieux et 'lemmatizer'
nlp = spacy.load("fr_core_news_lg", disable=["parser"])
nlp.max_length = 2_000_000  

#on convertit en liste
textes_bruts = df["texte"].astype(str).tolist()

textes_nettoyes = []

# Utilisation de nlp.pipe pour traiter les textes par blocs (très rapide)
for doc in nlp.pipe(textes_bruts, batch_size=256, n_process=2): 
    tokens = [] # Liste pour stocker les tokens nettoyés
    
    for token in doc:
        lemme = token.lemma_.lower() # Obtenir le lemme du token en minuscules
        
        if (
            not token.is_stop # Ignorer les stop words spaCy par défaut
            and not token.is_punct # Ignorer la ponctuation
            and not token.like_num # Ignorer les chiffres
            and not token.is_space # Ignorer les espaces vides
            and token.ent_type_ not in ['PER', 'LOC', 'ORG'] # Ignorer les Personnages, Lieux et Organisations
            and token.pos_ in {"NOUN", "ADJ"}  # Garder les noms et les adjectifs
            and len(lemme) > 2 # Ignorer les mots de 1 ou 2 lettres
            and lemme not in stop_perso
        ):
            tokens.append(lemme)
            
    # Rejoindre les tokens validés et les ajouter à la liste finale
    textes_nettoyes.append(" ".join(tokens))

# Application de la liste nettoyée à la nouvelle colonne du DataFrame
df["phrases_lemm"] = textes_nettoyes

chemin_sortie = Path("..") /"data" /"2_processed" /"03_corpus_lematise_128.csv"
chemin_sortie.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(chemin_sortie, index=False, encoding="utf-8")

# Affichage du résultat
df[["phrases_lemm"]].head()

,phrases_lemm
0,hiver matin frais manteau brouillard saison so...
1,soir vent porte mur flamme lampe ennui morne g...
2,chambre bel toile blanc meuble simple luisant ...
3,lèvre cœur reine laurier songe daignion règle ...
4,-vou brun rieur moisson vendange épi grappe se...
